# Agentic Workflow

imports and envs

In [21]:
# agentic_workflow.py
# Import the following agents: ActionPlanningAgent, KnowledgeAugmentedPromptAgent, EvaluationAgent, RoutingAgent from the workflow_agents.base_agents module
from workflow_agents.base_agents import ActionPlanningAgent, KnowledgeAugmentedPromptAgent, EvaluationAgent, RoutingAgent
import os
from dotenv import load_dotenv

# Load the openai_api_key variable with your OpenAI API key
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

## Instantiate Agents

- Action Planning Agent
- Product Manager Knowledge Agent
- Product Manager Evaluation Agent
- Program Manager Knowledge Agent
- Program Manager Evaluation Agent
- Develeopment Engineer Knowledge Agent
- Develeopment Engineer Evaluation Agent

### Action Planning Agent

In [33]:
# Action Planning Agent
knowledge_action_planning ="""
Stories are defined from a product spec by identifying a persona, an action, and a desired outcome for each story
Each story represents a specific functionality of the product described in the specification.
Features are defined by grouping related user stories.
Tasks are defined for each story and represent the engineering work required to develop the product.
A development Plan for a product contains all these components
"""

# Instantiate an action_planning_agent using the 'knowledge_action_planning'
action_planning_agent = ActionPlanningAgent(
    openai_api_key = openai_api_key,
    knowledge = knowledge_action_planning)

### Product Manager Knowledge Agent

In [23]:
# Product Manager - Knowledge Augmented Prompt Agent
persona_product_manager = """
You are a Product Manager.
You, as a product manger, are responsible for defining the user stories for a product.

Return 8–12 user stories in the form:
"As a [user type], I want [action/feature] so that [benefit/jobs to be done]."
Do not include any functionality outside this product.

Specification:
[PASTE EMAIL ROUTER SPEC HERE]
"""

path_to_document_file = os.path.join(os.path.abspath("."), "Product-Spec-Email-Router.txt")
product_spec = open(path_to_document_file, "rt").read()

knowledge_product_manager = f"""
Stories are defined by writing sentences with a persona, an action, and a desired outcome.
The sentences always start with: As a
Write several stories for the product spec based on your knowledge.
The personas are the different users of the product.

Your knowledge is defined as below:
{product_spec}
"""

# Instantiate a product_manager_knowledge_agent using 'persona_product_manager'
# and the completed 'knowledge_product_manager'
product_manager_knowledge_agent = KnowledgeAugmentedPromptAgent(
    openai_api_key = openai_api_key,
    persona = persona_product_manager,
    knowledge = knowledge_product_manager
)

### Product Manager Evaluation Agent

In [24]:
# Product Manager - Evaluation Agent
# Define the persona and evaluation criteria for a Product Manager evaluation agent
# and instantiate it as product_manager_evaluation_agent.
# This agent will evaluate the product_manager_knowledge_agent.
# The evaluation_criteria should specify the expected structure for user stories
# (e.g., "As a [type of user], I want [an action or feature] so that [benefit/value].").

persona_product_manager_evaluation_agent = "You are an evaluation agent that checks the answers of other worker agents"
evaluation_criteria_product_manager_evaluatoin_agent = "The answer should be stories that follow the following structure: As a [type of user], I want [an action or feature] so that [benefit/value]."
product_manager_evaluation_agent = EvaluationAgent(
    openai_api_key = openai_api_key,
    persona = persona_product_manager_evaluation_agent,
    evaluation_criteria = evaluation_criteria_product_manager_evaluatoin_agent,
    worker_agent = product_manager_knowledge_agent,
    max_interactions=3
)

### Program Manager Knowledge Agent

In [25]:
# Program Manager - Knowledge Augmented Prompt Agent
persona_program_manager = """
You are a Program Manager, you are responsible for defining the features for a product.
"""

knowledge_program_manager = "Features of a product are defined by organizing similar user stories into cohesive groups."

# Instantiate a program_manager_knowledge_agent using 'persona_program_manager' and 'knowledge_program_manager'
program_manager_knowledge_agent = KnowledgeAugmentedPromptAgent(
    openai_api_key= openai_api_key,
    persona = persona_program_manager,
    knowledge = knowledge_program_manager
)

### Program Manager Evaluation Agent

In [26]:
# Program Manager - Evaluation Agent

# Instantiate a program_manager_evaluation_agent using 'persona_program_manager_eval' and the evaluation criteria below.
#                      "The answer should be product features that follow the following structure: " \
#                      "Feature Name: A clear, concise title that identifies the capability\n" \
#                      "Description: A brief explanation of what the feature does and its purpose\n" \
#                      "Key Functionality: The specific capabilities or actions the feature provides\n" \
#                      "User Benefit: How this feature creates value for the user"
# For the 'agent_to_evaluate' parameter, refer to the provided solution code's pattern.
persona_program_manager_eval = """
You are an evaluation agent that checks the answers of other worker agents.
"""
evaluation_criteria_program_manager_evaluation_agent = """
The output should be product features that follow the following structure:

[Structure]
- Feature Name: [Insert a clear, concise title that identifies the capability] in string
- Description: [Insert a brief explanation of what the feature does and its purpose] in string
- Key Functionality: [Insert the specific capabilities or actions the feature provides] in string
- User Benefit: [Insert how this feature creates value for the user] in string
- User Story: [Insert a user story that describes how the feature will be used] in string

Explanation of each items in the structure.
"""
program_manager_evaluation_agent = EvaluationAgent(
    openai_api_key = openai_api_key,
    persona = persona_program_manager_eval,
    evaluation_criteria = evaluation_criteria_program_manager_evaluation_agent,
    worker_agent = program_manager_knowledge_agent,
    max_interactions = 3
)

### Develeopment Engineer Knowledge Agent

In [27]:
# Development Engineer - Knowledge Augmented Prompt Agent
persona_dev_engineer = "You are a Development Engineer, you are responsible for defining the development tasks for a product."

knowledge_dev_engineer = """
Development tasks are defined by identifying what needs to be built to implement each user story.
"""

# Instantiate a development_engineer_knowledge_agent using 'persona_dev_engineer' and 'knowledge_dev_engineer'
development_engineer_knowledge_agent = KnowledgeAugmentedPromptAgent(
    openai_api_key = openai_api_key,
    persona = persona_dev_engineer,
    knowledge = knowledge_dev_engineer,
)

### Develeopment Engineer Evaluation Agent

In [28]:
# Development Engineer - Evaluation Agent
persona_dev_engineer_eval = "You are an evaluation agent that checks the answers of other worker agents."
evaluation_criteria_development_engineer_evaluation_agent = """
The answer should be tasks following this exact structure:

[structure]
Task ID: [Insert unique identifier]
Task Title: [Insert brief description of the specific development work] in string format
Related User Story: [Insert reference to the parent user story] in string format
Description: [Insert detailed explanation of the technical work required] in string format
Acceptance Criteria: [Insert specific requirements that must be met for completion] in string format
Estimated Effort: [Insert time or complexity estimation] in string format
Dependencies: [Insert Any tasks that must be completed first] in string format
"""

# Instantiate a development_engineer_evaluation_agent using 'persona_dev_engineer_eval' and the evaluation criteria below.
#                      "The answer should be tasks following this exact structure: " \
#                      "Task ID: A unique identifier for tracking purposes\n" \
#                      "Task Title: Brief description of the specific development work\n" \
#                      "Related User Story: Reference to the parent user story\n" \
#                      "Description: Detailed explanation of the technical work required\n" \
#                      "Acceptance Criteria: Specific requirements that must be met for completion\n" \
#                      "Estimated Effort: Time or complexity estimation\n" \
#                      "Dependencies: Any tasks that must be completed first"
# For the 'agent_to_evaluate' parameter, refer to the provided solution code's pattern.
development_engineer_evaluation_agent = EvaluationAgent(
    openai_api_key = openai_api_key,
    persona = persona_dev_engineer_eval,
    evaluation_criteria = evaluation_criteria_development_engineer_evaluation_agent,
    worker_agent = development_engineer_knowledge_agent,
    max_interactions = 5
)

### Routing Agent

In [48]:
# Routing Agent
# Instantiate a routing_agent. You will need to define a list of agent dictionaries (routes) for Product Manager, Program Manager, and Development Engineer. Each dictionary should contain 'name', 'description', and 'func' (linking to a support function). Assign this list to the routing_agent's 'agents' attribute.
routing_agent = RoutingAgent(
    openai_api_key=openai_api_key,
)

# Job function persona support functions
# Define the support functions for the routes of the routing agent (e.g., product_manager_support_function, program_manager_support_function, development_engineer_support_function).
# Each support function should:
#   1. Take the input query (e.g., a step from the action plan).
#   2. Get a response from the respective Knowledge Augmented Prompt Agent.
#   3. Have the response evaluated by the corresponding Evaluation Agent.
#   4. Return the final validated response.
# Updated for reusability of the code
def support_function(query: str,
                    eval_agent: EvaluationAgent):
    # Following commented out as the evaluation agent implementation execute this
    # Rather if we call here, initial query is againg executed by a worker agent
    # in the evaluation agent.
    # See the worker agent is set in each evaluation agent, when they are intantiated.
    #initial_response = knowledge_agent.respond(query)
    evaluation_response = eval_agent.evaluate(query)
    return evaluation_response["final_response"]

routing_agent.agents = [
    {
        "name": "Product Manager",
        "description": "Responsible for defining user persona and users stories of a product.",
        "func": lambda query: support_function(query, product_manager_evaluation_agent)
    },
    {
        "name": "Program Manager",
        "description": "Responsible for defining product features and user benefits based on user persona and stories.",
        "func": lambda query: support_function(query, program_manager_evaluation_agent)
    },
    {
        "name": "Development Engineer",
        "description": "responsible for planning the development and implementation for the product.",
        "func": lambda query: support_function(query, development_engineer_evaluation_agent)
    }
]

## Start Test

In [49]:
# Run the workflow
print("\n*** Workflow execution started ***\n")
# Workflow Prompt
# ****
workflow_prompt = "What would the development tasks for this product be?"
# ****
print(f"Task to complete in this workflow, workflow prompt = {workflow_prompt}")

print("\nDefining workflow steps from the workflow prompt")


*** Workflow execution started ***

Task to complete in this workflow, workflow prompt = What would the development tasks for this product be?

Defining workflow steps from the workflow prompt


In [50]:
# Implement the workflow.
#   1. Use the 'action_planning_agent' to extract steps from the 'workflow_prompt'.
steps = action_planning_agent.extract_steps_from_prompt(workflow_prompt)

print("Action planned by the planning agent")
for step in steps:
    print(step)

Action planned by the planning agent
1. Define user stories based on product spec
2. Group related user stories into features
3. Define tasks for each user story
4. Create a development plan outlining the sequence of tasks
5. Assign tasks to team members
6. Set deadlines for each task
7. Monitor progress and adjust plan as needed
8. Test the product to ensure it meets the requirements
9. Implement any necessary changes based on testing feedback
10. Deploy the product for use by customers


In [51]:
#   2. Initialize an empty list to store 'completed_steps'.
#   3. Loop through the extracted workflow steps:
#      a. For each step, use the 'routing_agent' to route the step to the appropriate support function.
#      b. Append the result to 'completed_steps'.
#      c. Print information about the step being executed and its result.
def exec_step(step: str):
    print(f"----------------------\nstep: {step}\n----------------------\n")
    result = routing_agent.route(step)
    print(f"----------------------\nresult: {result}\n----------------------\n")
    return result

completed_steps = [exec_step(step) for step in steps]

#   4. After the loop, print the final output of the workflow (the last completed step).
print(f"~~~~~~~~~~~~~~~~~~~\nfinal output\n~~~~~~~~~~~~~~~~~~~\n ")
for step in completed_steps:
    print(step + "\n\n")

----------------------
step: 1. Define user stories based on product spec
----------------------

0.631374276451842
0.6136283292239135
0.3391034553814684
[Router] Best agent: Product Manager (score=0.631)

--- Interaction 1 ---
 Step 1: Worker agent generates a response to the prompt
Prompt:
1. Define user stories based on product spec
Worker Agent Response:
As a Customer Support Representative, I want the Email Router system to automate responses to routine inquiries so that I can focus on addressing complex customer issues efficiently.

As a Subject Matter Expert (SME), I want the Email Router system to intelligently route complex inquiries to me based on content analysis so that I can provide specialized assistance effectively.

As an IT Administrator, I want the Email Router system to provide a comprehensive dashboard for monitoring system performance and metrics so that I can ensure the system operates smoothly.

As a Customer Support Representative, I want the Email Router system